In [8]:
!pip install -Uqqq pip --progress-bar off
!pip install -qqq transformers==5.3.0 --progress-bar off
!pip install -qqq torch==2.10.0 --progress-bar off
!pip install -qqq sentence-transformers==5.2.3 --progress-bar off

In [26]:
!gdown 1_C7S8yP-YzrYyaY2E_c6Tp3_-bfeffA2

Downloading...
From: https://drive.google.com/uc?id=1_C7S8yP-YzrYyaY2E_c6Tp3_-bfeffA2
To: /content/modernbert-blog.md
100% 32.9k/32.9k [00:00<00:00, 63.3MB/s]


In [3]:
import torch
from transformers import pipeline

pipe = pipeline(
    "fill-mask",
    model="answerdotai/ModernBERT-base",
    dtype=torch.bfloat16,
    device_map="auto",
)

Loading weights:   0%|          | 0/137 [00:00<?, ?it/s]

In [5]:
pipe.model.config

ModernBertConfig {
  "architectures": [
    "ModernBertForMaskedLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 50281,
  "classifier_activation": "gelu",
  "classifier_bias": false,
  "classifier_dropout": 0.0,
  "classifier_pooling": "mean",
  "cls_token_id": 50281,
  "decoder_bias": true,
  "deterministic_flash_attn": false,
  "dtype": "bfloat16",
  "embedding_dropout": 0.0,
  "eos_token_id": 50282,
  "global_attn_every_n_layers": 3,
  "gradient_checkpointing": false,
  "hidden_activation": "gelu",
  "hidden_size": 768,
  "initializer_cutoff_factor": 2.0,
  "initializer_range": 0.02,
  "intermediate_size": 1152,
  "layer_norm_eps": 1e-05,
  "layer_types": [
    "full_attention",
    "sliding_attention",
    "sliding_attention",
    "full_attention",
    "sliding_attention",
    "sliding_attention",
    "full_attention",
    "sliding_attention",
    "sliding_attention",
    "full_attention",
    "sliding_attention",
    "sliding_attention",
    "full_

## Masking

In [4]:
%%time
input_text = "The AI engineer deployed the model to [MASK]."
results = pipe(input_text)

for res in results:
    print(f"Token: {res['token_str']}, Score: {res['score']:.4f}")

Token:  work, Score: 0.0574
Token:  production, Score: 0.0396
Token:  Azure, Score: 0.0255
Token:  AWS, Score: 0.0211
Token:  Facebook, Score: 0.0198
CPU times: user 142 ms, sys: 117 ms, total: 259 ms
Wall time: 650 ms


## Embeddings

In [9]:
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim

model = SentenceTransformer("Alibaba-NLP/gte-modernbert-base")
query = "How does Alternating Attention work?"

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/205 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/298M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

In [13]:
from pathlib import Path

blog = Path("modernbert-blog.md").read_text()
print(blog[:250])


# Finally, a Replacement for BERT

## TL;DR

This blog post introduces [ModernBERT](https://huggingface.co/collections/answerdotai/modernbert-67627ad707a4acbf33c41deb), a family of state-of-the-art encoder-only models representing improvements over 


In [21]:
documents = []
for part in blog.split("#"):
    doc = part.strip()
    if doc:
        print(doc)
        print("-" * 100)
        documents.append(doc)

Finally, a Replacement for BERT
----------------------------------------------------------------------------------------------------
TL;DR

This blog post introduces [ModernBERT](https://huggingface.co/collections/answerdotai/modernbert-67627ad707a4acbf33c41deb), a family of state-of-the-art encoder-only models representing improvements over older generation encoders across the board, with a **8192** sequence length, better downstream performance and much faster processing. 

ModernBERT is available as a *slot-in* replacement for any BERT-like models, with both a **base** (149M params) and **large** (395M params) model size.

<details><summary>Click to see how to use these models with <code>transformers</code></summary>

ModernBERT will be included in v4.48.0 of `transformers`. Until then, it requires installing transformers from main:

```sh
pip install git+https://github.com/huggingface/transformers.git
```

Since ModernBERT is a Masked Language Model (MLM), you can use the `fill-mas

In [25]:
len(documents)

23

In [24]:
%%time
query_emb = model.encode(query)
doc_embs = model.encode(documents)

scores = cos_sim(query_emb, doc_embs)[0]

CPU times: user 987 ms, sys: 1.91 ms, total: 989 ms
Wall time: 1.02 s


In [23]:
top_k = torch.topk(scores, k=5)
for score, idx in zip(top_k.values, top_k.indices):
    print(
        f"Doc {idx.item()} | Score: {score:.3f} | Snippet: {documents[idx.item()][:100]}..."
    )

Doc 14 | Score: 0.681 | Snippet: Global and Local Attention

One of ModernBERT’s most impactful features is **Alternating** **Attenti...
Doc 0 | Score: 0.608 | Snippet: Finally, a Replacement for BERT...
Doc 20 | Score: 0.604 | Snippet: The tricks, it’s all about the tricks\!

If you’ve made it this far into this announcement, you’re p...
Doc 13 | Score: 0.582 | Snippet: Upgrading a Honda Civic for the Race Track

We’ve covered this already: encoders are no Ferraris, an...
Doc 19 | Score: 0.578 | Snippet: Process

We stick to the original BERT’s training recipe, with some slight upgrades inspired by subs...
